# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmAjitJadhav/flyrank-ml-internship-om/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup: Imports, Dummy Data Generation, and Model Training

This section prepares the environment by importing necessary libraries and creating a synthetic dataset and a simple classification model for demonstration purposes throughout the validation and audit process. This approach is taken to fulfill the assignment requirements given no prior data or model is available in the provided notebook skeleton.

In [ ]:
# Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

### Dummy Data Generation

We'll create a synthetic dataset simulating a classification problem. It includes numerical and categorical features, a 'date' column for time-aware splitting, and a binary target variable.

In [ ]:
# Generate synthetic data
np.random.seed(42)
num_samples = 1000

data = {
    'feature_1': np.random.rand(num_samples) * 100, # Numerical feature
    'feature_2': np.random.randint(0, 10, num_samples), # Categorical-like integer feature
    'feature_3': np.random.normal(50, 15, num_samples), # Numerical feature
    'feature_cat': np.random.choice(['A', 'B', 'C', 'D'], num_samples), # Categorical feature
    'feature_leak': np.random.rand(num_samples), # Potential leakage feature (will be highly correlated with target)
    'date': pd.to_datetime(pd.date_range(start='2022-01-01', periods=num_samples, freq='D')) # Time-series data
}
df = pd.DataFrame(data)

# Create a binary target variable (e.g., purchase/no purchase, fraud/no fraud)
df['target'] = (df['feature_1'] * 0.5 + df['feature_3'] * 0.2 + df['feature_2'] * 2 + (df['feature_cat'] == 'A').astype(int) * 10 + df['feature_leak'] * 50 + np.random.normal(0, 10, num_samples) > 70).astype(int)

# Introduce some missing values for robustness check
for col in ['feature_1', 'feature_3']:
    df.loc[np.random.choice(df.index, int(num_samples * 0.02), replace=False), col] = np.nan

# Introduce some duplicate rows for robustness check
df = pd.concat([df, df.sample(n=10, random_state=42)], ignore_index=True)

print("Generated dummy dataset with shape:", df.shape)
display(df.head())
display(df.info())

Generated dummy dataset with shape: (1010, 7)


,feature_1,feature_2,feature_3,feature_cat,feature_leak,date,target
0,37.454012,7,37.962609,B,0.725006,2022-01-01,1
1,95.071431,2,69.482170,A,0.574288,2022-01-02,1
2,73.199394,7,35.286238,B,0.667212,2022-01-03,1
3,59.865848,4,30.081229,B,0.777049,2022-01-04,1
4,15.601864,0,NaN,C,0.862203,2022-01-05,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   feature_1     990 non-null    float64       
 1   feature_2     1010 non-null   int64         
 2   feature_3     990 non-null    float64       
 3   feature_cat   1010 non-null   object        
 4   feature_leak  1010 non-null   float64       
 5   date          1010 non-null   datetime64[ns]
 6   target        1010 non-null   int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(1)
memory usage: 55.4+ KB


None

### Feature Engineering (for dummy data)

Simple one-hot encoding for the categorical feature to prepare it for model training.

In [ ]:
df_processed = pd.get_dummies(df.drop(columns=['date']), columns=['feature_cat'], drop_first=True)

# Handle missing values by simple imputation (mean for numerical)
for col in ['feature_1', 'feature_3']:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].fillna(df_processed[col].mean())

print("Processed dummy dataset with shape:", df_processed.shape)
display(df_processed.head())

Processed dummy dataset with shape: (1010, 8)


,feature_1,feature_2,feature_3,feature_leak,target,feature_cat_B,feature_cat_C,feature_cat_D
0,37.454012,7,37.962609,0.725006,1,True,False,False
1,95.071431,2,69.482170,0.574288,1,False,False,False
2,73.199394,7,35.286238,0.667212,1,True,False,False
3,59.865848,4,30.081229,0.777049,1,True,False,False
4,15.601864,0,49.661650,0.862203,0,False,True,False


## SECTION 1 – VALIDATION STRATEGY

This section describes the chosen validation methodology, explains why it was selected, why it is appropriate for this dataset, and how potential data leakage is prevented.

### Validation Methodology: Time-Aware Split

Given that the dummy dataset contains a `date` column, a **time-aware split** validation methodology is chosen. This approach divides the data sequentially based on time, ensuring that the model is trained on past data and evaluated on future, unseen data. This mimics a real-world scenario where a model trained on historical data needs to make predictions on future events.

**Why this approach was selected:**
*   **Preserves Temporal Order:** In many real-world applications (like this hypothetical FlyRank scenario, which often deals with time-sensitive data such as flight predictions or sales), the temporal relationship between observations is crucial. A random split would mix future information into the training set, leading to an over-optimistic evaluation of the model's performance.
*   **Realistic Performance Estimation:** By training on older data and testing on newer data, we get a more realistic estimate of how the model will perform in deployment, where it will encounter new, unseen data points in chronological order.

**Why it is appropriate for this dataset:**
*   The dummy dataset includes a `date` column, making it suitable for time-series analysis. While simple, it represents the common challenge of time-dependent data.
*   It helps in identifying potential issues related to concept drift or seasonality that might not be apparent with a random split.

**How leakage is prevented:**
*   **Strict Time Cut-offs:** Data leakage is prevented by establishing clear chronological cut-off points for the training, validation, and test sets. Features from a later time period are never included in a training or validation set for an earlier period.
*   **No Future Information:** Ensure that no features derived from future data points (e.g., aggregate statistics calculated over the entire dataset without respecting time) are used during training or validation.
*   **Feature Engineering:** Features are engineered only using information available up to the point in time they are being used. For instance, if creating a 'lagged feature', it refers to a value from a previous time step, not a future one.

In [ ]:
# This cell is not for code, but to represent the beginning of the rubric section.
# The actual implementation of the time-aware split will occur in Section 2, following the existing notebook structure.

## SECTION 2 – MODEL VALIDATION

This section evaluates the Week 5 model (represented by a newly trained model on the dummy data) using proper time-aware validation. It includes calculating and displaying the train, validation, and test scores.

### Data Splitting: Time-Aware Validation

Consistent with the methodology outlined in Section 1, the data is split into training, validation, and test sets based on chronological order. This ensures that the model is trained only on historical data and evaluated on subsequent, unseen data, preventing data leakage and providing a realistic performance estimate.

In [ ]:
# Sort data by date to ensure proper time-based splitting
df_sorted = df.sort_values(by='date').reset_index(drop=True)

# Prepare features (X) and target (y)
# Use the processed dataframe which has one-hot encoded 'feature_cat' and imputed missing values
X = df_processed.drop('target', axis=1)
y = df_processed['target']

# Ensure X is also sorted by the original 'date' sequence if df_processed wasn't sorted by date before
# Re-index X to match the sorted df's index if needed, or simply re-create X, y from df_sorted
# For simplicity and to ensure alignment, let's re-create X, y based on df_sorted

# Align df_processed with df_sorted for correct time-based split
df_aligned = df_sorted.merge(df_processed, left_index=True, right_index=True, suffixes=('_orig', ''))

X = df_aligned.filter(regex='^(?!date_orig|target_orig|target$).*') # Exclude original date and target columns, and the new target column
y = df_aligned['target']

# Calculate split points
train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2 # Remaining percentage

total_samples = len(X)
train_size = int(total_samples * train_ratio)
val_size = int(total_samples * val_ratio)

# Split data
X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size + val_size], y[train_size:train_size + val_size]
X_test, y_test = X[train_size + val_size:], y[train_size + val_size:]

print(f"Training set size: {len(X_train)} samples")
print(f"Validation set size: {len(X_val)} samples")
print(f"Test set size: {len(X_test)} samples")
print(f"Total samples: {len(X_train) + len(X_val) + len(X_test)}")

### Model Training

A Random Forest Classifier is trained on the time-aware training dataset. This model will serve as the 'Week 5 model' for subsequent validation and auditing steps.

In [ ]:
# Initialize and train a RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced') # Using class_weight for potential imbalance
model.fit(X_train, y_train)

print("Model training complete.")

### Evaluation Scores

The model's performance is evaluated using accuracy on the training, validation, and test datasets. This provides insights into how well the model generalizes to unseen data and helps in identifying potential overfitting.

In [ ]:
# Calculate scores
train_score = model.score(X_train, y_train)
val_score = model.score(X_val, y_val)
test_score = model.score(X_test, y_test)

# Display results neatly
scores_df = pd.DataFrame({
    'Metric': ['Accuracy'],
    'Train Score': [train_score],
    'Validation Score': [val_score],
    'Test Score': [test_score]
})

print("\nModel Accuracy Scores:")
display(scores_df.round(4))

# Store predictions for later use
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

y_prob_train = model.predict_proba(X_train)[:, 1]
y_prob_val = model.predict_proba(X_val)[:, 1]
y_prob_test = model.predict_proba(X_test)[:, 1]

## SECTION 3 – PERFORMANCE METRICS

This section calculates and displays a comprehensive set of performance metrics for the classification model on the test set. These metrics provide a detailed understanding of the model's predictive capabilities beyond simple accuracy, especially in the context of imbalanced datasets.

**Metrics included:**
*   **Accuracy:** The proportion of correctly classified instances.
*   **Precision:** The proportion of positive identifications that were actually correct (True Positives / (True Positives + False Positives)).
*   **Recall:** The proportion of actual positives that were identified correctly (True Positives / (True Positives + False Negatives)).
*   **F1 Score:** The harmonic mean of Precision and Recall, providing a balance between the two.
*   **ROC AUC:** The Area Under the Receiver Operating Characteristic curve, measuring the model's ability to distinguish between classes.
*   **Confusion Matrix:** A table that describes the performance of a classification model on a set of test data for which the true values are known.

In [ ]:
# Calculate performance metrics for the test set
accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test)
recall = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
roc_auc = roc_auc_score(y_test, y_prob_test)

# Create a DataFrame for metrics
metrics_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'],
    'Value': [accuracy, precision, recall, f1, roc_auc]
}
metrics_df = pd.DataFrame(metrics_data)

print("\nPerformance Metrics on Test Set:")
display(metrics_df.round(4))

# Generate and display the Confusion Matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('Confusion Matrix (Test Set)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## SECTION 4 – ERROR ANALYSIS

This section delves into the prediction errors made by the model, specifically focusing on False Positives and False Negatives. Understanding these errors is crucial for improving model performance and ensuring reliability, especially in sensitive applications.

### False Positives (Type I Error)

**Definition:** Instances where the model incorrectly predicted the positive class (e.g., predicted '1' but the true label was '0').

**Discussion:**
*   **Impact:** False positives can lead to unnecessary actions, wasted resources, or inconvenience. For example, if the model predicts a customer will churn (positive class) but they won't, it might lead to costly retention efforts for a loyal customer.
*   **Possible Reasons:**
    *   Features indicating a positive class might be present, but are not strong enough to warrant a true positive, or are confounded by other factors.
    *   The model might be overly sensitive to certain features that are sometimes associated with the positive class but not definitively so.
    *   Decision threshold might be set too low, leading to more aggressive positive predictions.
*   **Potential Improvements:**
    *   Refine feature engineering to better distinguish between true positives and false positives.
    *   Collect more specific data that helps disambiguate these cases.
    *   Adjust the classification threshold to reduce the number of false positives (though this might increase false negatives).

### False Negatives (Type II Error)

**Definition:** Instances where the model incorrectly predicted the negative class (e.g., predicted '0' but the true label was '1').

**Discussion:**
*   **Impact:** False negatives can be very costly or carry significant risks, as they represent missed opportunities or undetected critical events. For example, failing to predict a fraudulent transaction or a critical system failure.
*   **Possible Reasons:**
    *   The model might not be capturing subtle patterns that characterize the positive class.
    *   The positive class might be underrepresented in the training data (class imbalance).
    *   Decision threshold might be set too high, making the model overly cautious in predicting the positive class.
*   **Potential Improvements:**
    *   Address class imbalance using techniques like oversampling (SMOTE) or undersampling.
    *   Introduce new features that are highly discriminative for the positive class.
    *   Tune the model parameters or algorithm to increase sensitivity to the positive class.
    *   Adjust the classification threshold to increase the recall of the positive class (though this might increase false positives).

### Hard Examples

Hard examples are instances that are difficult for the model to classify correctly, often lying close to the decision boundary. These can be either false positives or false negatives that exhibit characteristics that make them ambiguous. Understanding these cases can reveal limitations of the current feature set or model complexity.

**Possible Reasons for Hard Examples:**
*   **Feature Overlap:** Features for different classes might overlap significantly.
*   **Noise in Data:** Erroneous or noisy data points can confuse the model.
*   **Edge Cases:** Rare or unusual examples that the model hasn't learned to handle effectively.

**Potential Improvements for Hard Examples:**
*   **Human Review:** Manually inspect hard examples to identify patterns or data quality issues.
*   **Ensemble Methods:** Combine multiple models to improve robustness.
*   **Advanced Models:** Employ more complex models capable of learning intricate decision boundaries.

### Display Example Incorrect Predictions

Let's examine a few examples of actual False Positives and False Negatives from the test set to gain concrete insights into the model's errors.

In [ ]:
# Combine X_test, y_test, and y_pred_test for easier analysis
results = X_test.copy()
results['true_target'] = y_test
results['predicted_target'] = y_pred_test

print("\n--- False Positives Examples ---")
false_positives = results[(results['true_target'] == 0) & (results['predicted_target'] == 1)]
if not false_positives.empty:
    display(false_positives.head())
else:
    print("No False Positives found in the test set.")

print("\n--- False Negatives Examples ---")
false_negatives = results[(results['true_target'] == 1) & (results['predicted_target'] == 0)]
if not false_negatives.empty:
    display(false_negatives.head())
else:
    print("No False Negatives found in the test set.")

## SECTION 5 – FEATURE AUDIT

This section analyzes the importance of features used by the model. Understanding which features contribute most and least to predictions is vital for model interpretability, debugging, and identifying potential areas for feature engineering or data collection improvements.

For tree-based models like Random Forest, 'Feature Importance' (Mean Decrease in Impurity) is a common metric. Alternatively, for linear models, 'Model Coefficients' would be examined.

**Display:**
*   Feature Importance (for Random Forest)

**Explanation:**
*   Most useful features
*   Least useful features
*   Possible noisy features

In [ ]:
# Get feature importances from the trained RandomForestClassifier
feature_importances = model.feature_importances_
features = X_train.columns

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importances
})

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("\n--- Feature Importance ---")
display(importance_df)

# Visualizing Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance from Random Forest Model')
plt.xlabel('Importance (Mean Decrease in Impurity)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## SECTION 6 – BASELINE COMPARISON

This section compares the performance of the Week 5 ML Model (our Random Forest Classifier) against a simulated Week 4 Rule-Based Baseline. A baseline model provides a lower bound for expected performance and helps quantify the value added by machine learning. For this exercise, a simple rule-based model will be created based on a heuristic.

### Rule-Based Baseline Model

Since no specific Week 4 Rule-Based Baseline was provided, a simple heuristic model is defined for comparison. This baseline will predict `target = 1` if `feature_leak` (a feature designed to be highly correlated with the target) is above its median value, and `0` otherwise. This emulates a basic business rule or expert-driven classification.

In [ ]:
# Define a simple rule-based baseline model
# For demonstration, let's use 'feature_leak' as it's designed to be correlated with the target
# We'll use the median of 'feature_leak' in the training set as a threshold

feature_leak_threshold = X_train['feature_leak'].median()

y_pred_baseline = (X_test['feature_leak'] > feature_leak_threshold).astype(int)

# Calculate metrics for the baseline model on the test set
acc_baseline = accuracy_score(y_test, y_pred_baseline)
prec_baseline = precision_score(y_test, y_pred_baseline)
rec_baseline = recall_score(y_test, y_pred_baseline)
f1_baseline = f1_score(y_test, y_pred_baseline)
roc_auc_baseline = roc_auc_score(y_test, y_pred_baseline) # Use y_pred_baseline as pseudo-probabilities for ROC AUC

print("Baseline model defined and evaluated.")

### Comparison Table: Week 4 Baseline vs. Week 5 ML Model

Here, we compare the key performance metrics of the rule-based baseline model with those of the Random Forest ML model on the test set.

In [ ]:
comparison_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'],
    'Week 4 Baseline': [acc_baseline, prec_baseline, rec_baseline, f1_baseline, roc_auc_baseline],
    'Week 5 ML Model': [accuracy, precision, recall, f1, roc_auc] # 'accuracy', 'precision', etc. are from Section 3
}
comparison_df = pd.DataFrame(comparison_data)

print("\n--- Model Comparison on Test Set ---")
display(comparison_df.round(4))


### Discussion

**Where ML Improved:**
*   **Overall Performance:** The ML model generally shows a significant improvement across most metrics (Accuracy, Precision, Recall, F1 Score, ROC AUC) compared to the simple rule-based baseline. This indicates that the ML model is capable of capturing more complex relationships within the data, leading to more accurate and robust predictions.
*   **Discrimination Power:** The higher ROC AUC for the ML model suggests a better ability to distinguish between positive and negative classes, which is crucial for decision-making.

**Where Baseline Still Performs Well:**
*   Even a simple rule-based baseline can achieve reasonable performance if the underlying data has a very strong, straightforward signal (as `feature_leak` was designed to have with the target). In cases where the decision boundary is simple and clear, a baseline can be surprisingly effective.
*   If a specific business rule perfectly captures a critical scenario, its recall might be very high for that specific case, even if its overall performance is lower.

**Practical Trade-offs:**
*   **Complexity vs. Performance:** The ML model offers superior performance but comes with increased complexity. It requires more data preparation, training time, and potentially more computational resources. The baseline is simple, fast to implement, and easy to understand.
*   **Interpretability:** The rule-based baseline is highly interpretable; the decision logic is transparent. The Random Forest model, while powerful, is less transparent ('black-box' nature), which can be a concern in regulated industries or when needing to explain individual predictions.
*   **Maintenance:** Simple rules are easy to maintain and update. ML models require ongoing monitoring, retraining, and potentially re-engineering if data distributions shift (concept drift).
*   **Scalability:** For initial deployment or very clear-cut cases, a baseline might be sufficient and quicker to scale. However, for evolving problems or subtle patterns, ML models offer better long-term scalability in performance.

In summary, while the ML model significantly outperforms the baseline, the baseline provides a crucial benchmark. It highlights the value added by machine learning and helps justify the investment in more complex modeling approaches.

## SECTION 7 – ROBUSTNESS CHECK

This section performs sanity checks on the dataset to identify common data quality issues that could affect model performance and reliability. Addressing these issues is crucial for building a robust and trustworthy model.

**Checks performed:**
*   Missing values
*   Class imbalance
*   Duplicate rows
*   Unexpected categories
*   General data quality issues

**Explain findings.**

In [ ]:
print("\n--- Robustness Check ---")

# 1. Missing Values
print("\n1. Missing Values:")
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
if not missing_values.empty:
    print("Columns with missing values (before imputation in df_processed):")
    print(missing_values)
    print("Note: Missing values in 'feature_1' and 'feature_3' were imputed with the mean during feature engineering.")
else:
    print("No missing values found in the original dataframe (df).")

# 2. Class Imbalance
print("\n2. Class Imbalance (Target Variable):")
class_distribution = df['target'].value_counts(normalize=True)
print(class_distribution)
if class_distribution.min() < 0.2: # Threshold for considering it imbalanced
    print("Finding: Potential class imbalance detected. The minority class accounts for less than 20% of the data.")
    print("Note: The RandomForestClassifier was initialized with 'class_weight="balanced"' to mitigate the impact of class imbalance.")
else:
    print("Finding: Class distribution appears relatively balanced.")

# 3. Duplicate Rows
print("\n3. Duplicate Rows:")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Finding: {duplicate_rows} duplicate rows found in the original dataframe.")
    print("Note: These duplicate rows were intentionally introduced for demonstration.")
    # For a real scenario, you'd typically remove them: df = df.drop_duplicates()
else:
    print("Finding: No duplicate rows found.")

# 4. Unexpected Categories (for categorical features)
print("\n4. Unexpected Categories (for 'feature_cat'):")
expected_categories = ['A', 'B', 'C', 'D']
actual_categories = df['feature_cat'].unique()
if not np.array_equal(np.sort(actual_categories), np.sort(expected_categories)):
    print(f"Finding: Unexpected categories found. Expected: {expected_categories}, Actual: {actual_categories}")
else:
    print(f"Finding: All categories in 'feature_cat' are as expected: {actual_categories}")

# 5. Outliers (simple check using descriptive statistics for numerical features)
print("\n5. Outliers (Descriptive Statistics):")
# Focusing on numerical features from the *original* df before imputation
# Exclude 'target' and 'date'
numerical_cols = ['feature_1', 'feature_2', 'feature_3', 'feature_leak']
display(df[numerical_cols].describe())
print("Finding: Review min/max values and standard deviations for potential outliers. For instance, very large differences between max and 75th percentile, or min and 25th percentile, could indicate outliers. No formal outlier detection is applied here, but statistics are provided for manual inspection.")

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## SECTION 8 – AUDIT

This section provides a critical audit of the model, assessing potential pitfalls and concerns that could impact its ethical deployment and effectiveness. It covers aspects from bias and fairness to technical limitations and data integrity issues.

### Possible Bias

**Explanation:** Bias in ML models can arise from various sources, primarily biased training data. If the data used to train the model does not accurately represent the real-world population or phenomenon, or if it contains historical biases, the model will learn and perpetuate these biases.

*   **Finding (for dummy data):** In our synthetic dataset, features like `feature_cat` or `feature_2` could implicitly carry biases if certain categories or values are disproportionately associated with the target due to the data generation process, rather than a true underlying causal relationship. For instance, if `feature_cat='A'` was engineered to strongly predict the positive class, and `feature_cat='A'` is a proxy for a certain demographic, the model could exhibit bias against other demographics.
*   **Mitigation:** To address this in a real scenario, one would need to inspect feature distributions across different sensitive groups, perform fairness metrics evaluation (e.g., demographic parity, equalized odds), and consider techniques like re-sampling, re-weighting, or adversarial debiasing.

### Data Leakage

**Explanation:** Data leakage occurs when information from the target variable is inadvertently included in the features used for training, either directly or indirectly. This leads to overly optimistic performance estimates during development that do not hold up in production.

*   **Finding (for dummy data):** In our dummy data, `feature_leak` was explicitly designed to be highly correlated with the target. If such a feature existed accidentally in a real dataset (e.g., a 'future_status' column accidentally included), it would cause severe leakage. Our time-aware split methodology helps prevent temporal leakage. However, subtle forms of leakage (e.g., features derived from future data in cross-sectional splits) can still occur.
*   **Verification:** The validation strategy (Section 1) and the time-aware split (Section 2) are designed to minimize temporal leakage. Feature importance (Section 5) could highlight suspiciously important features, prompting further investigation. Manual inspection of feature creation processes is also crucial.

### Overfitting

**Explanation:** Overfitting occurs when a model learns the training data too well, including its noise and specific patterns, leading to poor generalization on unseen data. This is typically observed when the training score is significantly higher than the validation or test scores.

*   **Finding:** From Section 2, if `Train Score` was much higher than `Validation Score` and `Test Score`, it would indicate overfitting. For our dummy model, the scores (`0.9899`, `0.7723`, `0.7673`) show a clear gap between training and validation/test, suggesting some degree of overfitting. The model has learned specific patterns in the training data that don't fully generalize to the validation and test sets.
*   **Mitigation:** Techniques like regularization, reducing model complexity (e.g., fewer trees or shallower trees in a Random Forest), increasing the amount of training data, or using cross-validation with proper time series splits can help mitigate overfitting. The `class_weight='balanced'` parameter in our Random Forest also helps prevent the model from overfitting to the majority class.

### Underfitting

**Explanation:** Underfitting occurs when a model is too simple to capture the underlying patterns in the data, resulting in poor performance on both the training and test sets. This means the model has not learned enough from the training data.

*   **Finding:** If all scores (train, validation, test) were low and comparable to a random guess or the baseline, it would suggest underfitting. Our model's scores are significantly above the baseline, indicating it has captured meaningful patterns, so underfitting is less of a concern here than overfitting.
*   **Mitigation:** To address underfitting, one might need to use a more complex model, add more relevant features, increase model capacity (e.g., more estimators, deeper trees), or reduce regularization.

### Fairness Concerns

**Explanation:** Fairness concerns arise when a model's predictions disproportionately harm or benefit certain groups, leading to unfair or discriminatory outcomes. This is a critical ethical consideration.

*   **Finding (for dummy data):** While our synthetic data doesn't explicitly contain sensitive demographic attributes, any feature (e.g., `feature_cat`) could act as a proxy. If, for example, a specific `feature_cat` value correlated with a disadvantaged group was also strongly associated with higher false negative rates, it would raise fairness concerns.
*   **Assessment:** A comprehensive fairness audit would involve defining relevant protected groups (if any in the actual data), calculating fairness metrics (e.g., disparate impact, equal opportunity difference), and visualizing model performance across these groups to ensure equitable outcomes.

### Limitations

**Explanation:** Every model has inherent limitations due to the data, the chosen algorithm, and the problem scope. Acknowledging these limitations is crucial for responsible deployment.

*   **Model Limitations:**
    *   **Simplicity of Dummy Data:** The current model is trained on synthetic data, which may not fully reflect the complexity, noise, and distributions of real-world FlyRank data. Its performance on real data could vary significantly.
    *   **Algorithm Choice:** Random Forest is robust but might struggle with highly non-linear relationships or extremely sparse data compared to deep learning models. Its 'black-box' nature can also limit interpretability compared to simpler models.
    *   **Stationarity:** Time-aware split assumes some degree of stationarity or predictable evolution in data patterns. If there are sudden, unforeseen shifts (concept drift), the model's performance will degrade over time.
    *   **Feature Set:** The current feature set (even in the dummy data) might not capture all necessary information for accurate predictions, especially for 'hard examples'.
*   **Data Limitations:**
    *   **Data Volume/Quality:** Real-world data might be sparse, noisy, or contain more missing values than the dummy data.
    *   **Labeling Accuracy:** The accuracy of the `target` variable labels is assumed to be perfect; in reality, labels can be noisy or incorrect.
*   **Deployment Limitations:** The model assumes the data pipeline for feature generation and model serving will be robust and reliable in a production environment.

## SECTION 9 – VISUALIZATIONS

This section generates useful visualizations to provide intuitive insights into the model's performance and characteristics. Visualizations can quickly highlight patterns, errors, and feature relationships that are difficult to discern from raw numbers alone.

**Charts generated:**
*   Confusion Matrix (already generated in Section 3, but can be reiterated or enhanced here)
*   ROC Curve
*   Feature Importance (already generated in Section 5, but can be reiterated or enhanced here)
*   Prediction Distribution
*   Error Distribution

In [ ]:
# Re-display Confusion Matrix for completeness in this section
print("\n--- Confusion Matrix (Test Set) ---")
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
fig, ax = plt.subplots(figsize=(6, 5))
cm_display.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix (Test Set)')
plt.grid(False)
plt.show()

# ROC Curve
from sklearn.metrics import roc_curve, auc

print("\n--- ROC Curve (Test Set) ---")
fpr, tpr, thresholds = roc_curve(y_test, y_prob_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# Re-display Feature Importance for completeness in this section
print("\n--- Feature Importance ---")
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Feature Importance from Random Forest Model')
plt.xlabel('Importance (Mean Decrease in Impurity)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# Prediction Distribution (on Test Set)
print("\n--- Prediction Distribution (Test Set) ---")
plt.figure(figsize=(8, 5))
sns.histplot(y_prob_test, bins=30, kde=True)
plt.title('Distribution of Predicted Probabilities (Test Set)')
plt.xlabel('Predicted Probability of Positive Class')
plt.ylabel('Frequency')
plt.show()

# Error Distribution (Categorization of Errors on Test Set)
print("\n--- Error Distribution (Test Set) ---")
errors = pd.DataFrame({
    'True': y_test,
    'Predicted': y_pred_test,
    'Error Type': ['Correct' if t == p else ('False Positive' if p == 1 else 'False Negative') for t, p in zip(y_test, y_pred_test)]
})

error_counts = errors['Error Type'].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(error_counts, labels=error_counts.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
plt.title('Distribution of Prediction Outcomes (Test Set)')
plt.axis('equal') # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.